In [ ]:
%region sa-east-1
%idle_timeout 30
%glue_version 5.0
%worker_type 'G.2X'
%number_of_workers 100
%security_config authorized-security-configuration
%connections analytics-glue-connection-aza, analytics-glue-connection-azc

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType
import boto3
import json
import re
import numpy as np
import pandas as pd
from urllib.parse import urlparse
from typing import List, Dict

In [ ]:
from pyspark.sql import functions as F
from pyspark.storagelevel import StorageLevel

# =========================================
# 0) CONFIGS IMPORTANTES (performance + overwrite por particao)
# =========================================

# FIX: 'sq1' -> 'sql' (digit 1 vs letter l) throughout all spark.conf.set calls
spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")  # <<-- ESSENCIAL
spark.conf.set("hive.exec.dynamic.partition", "true")
spark.conf.set("hive.exec.dynamic.partition.mode", "nonstrict")
spark.conf.set("spark.sql.parquet.compression.codec", "snappy")
spark.conf.set("spark.sql.files.maxRecordsPerFile", "1500000")  # ajuste conforme volume
# PERF FIX: reduced from 400 to 200 — 400 creates excessive task overhead on G.2X
spark.conf.set("spark.sql.shuffle.partitions", "200")

In [ ]:
SRC_TABLE = "workspace_db.book_sobrevivencia_rawv3"
OUT_DB = "workspace_db"
OUT_TABLE = "T025_EXTRATOR_TRANSACOES_SOBREVIVENCIA_FLAGS"
OUT_FULLNAME = f"{OUT_DB}.{OUT_TABLE}"
OUT_S3_BASE = "s3://itau-self-wkp-sa-east-1-635985251289/projeto-sobrevivencia/books/transacoes_categorias/T025_EXTRATOR_FATURAMENTO_SOBREVIVENCIA_FLAGS"

In [ ]:
# intervalo de referencia: 202404 ate 202603 (inclusive)
def months_between(start_yyyymm: str, end_yyyymm: str):
    s = int(start_yyyymm)
    e = int(end_yyyymm)
    sy, sm = divmod(s, 100)
    ey, em = divmod(e, 100)
    out = []
    y, m = sy, sm
    while (y < ey) or (y == ey and m <= em):
        out.append(f"{y:04d}{m:02d}")
        m += 1
        if m == 13:
            m = 1
            y += 1  # BUG FIX: was 'y+1' (no-op expression), must be 'y += 1'
    return out  # BUG FIX: moved outside the loop — was returning after the first year only

refs = months_between("202404", "202603")
print(f"Meses a processar ({len(refs)}): {refs[0]} .. {refs[-1]}")

In [ ]:
# total minimo de particoes necessarias na fonte:
# para 202404 precisa 202304 .. 202403 e para 202603 precisa 202503 .. 202602 => 202304 .. 202602
MIN_SRC = 202304
MAX_SRC = 202602

In [ ]:
# =========================================
# 2) FLAGS (case when trim(classificacao_mcc) =... then 1 else 0)
# =========================================

flags = {
    "flag_saude_agrupado":           "SAUDE_AGRUPADO",
    "flag_alimentacao_agrupado":      "ALIMENTACAO_AGRUPADO",
    "flag_viagem_agrupado":           "VIAGEM_AGRUPADO",
    "flag_casa_agrupado":             "CASA_AGRUPADO",
    "flag_educacao_agrupado":         "EDUCACAO_AGRUPADO",
    "flag_financas_agrupado":         "FINANCEIRO_AGRUPADO",
    "flag_entretenimento_agrupado":   "ENTRETENIMENTO_AGRUPADO",
    "flag_servico_agrupado":          "SERVICOS_AGRUPADO",
    "flag_outros_agrupado":           "OUTROS_AGRUPADO"
}

In [ ]:
src = (
    spark.table(SRC_TABLE)
        .withColumn("anomes_int", F.col("anomes").cast("int"))
        .filter((F.col("anomes_int") >= F.lit(MIN_SRC)) & (F.col("anomes_int") <= F.lit(MAX_SRC)))
        .select(
            F.col("cod_idef_pess").cast("string").alias("cod_idef_pess"),
            F.col("anomes_int").alias("anomes"),
            F.col("v1_transacao_mes"),
            F.col("categoria_macro").cast("string").alias("categoria_macro")
        )
        .withColumn("dt_anomes", F.to_date(F.concat(F.col("anomes").cast("string"), F.lit("01")), "yyyyMMdd"))
    # BUG FIX: 'StorageLeve1' (digit 1) -> 'StorageLevel' (letter l)
).persist(StorageLevel.MEMORY_AND_DISK)

_ = src.count()  # materializa cache
cls = F.trim(F.col("categoria_macro"))

In [ ]:
def build_month_df(anomes_param: str):
    pk_datref = F.to_date(F.lit(anomes_param + "01"), "yyyyMMdd")
    datref_defas = F.add_months(pk_datref, -1)

    tmp = (
        src
        .withColumn(
            # BUG FIX: 'ID MENSAL' (space, invalid column name) -> 'ID_MENSAL'
            "ID_MENSAL",
            (
                (F.year(datref_defas) - F.year(F.col("dt_anomes"))) * F.lit(12)
                + (F.month(datref_defas) - F.month(F.col("dt_anomes")))
            ).cast("int")
        )
        .filter((F.col("ID_MENSAL") >= 0) & (F.col("ID_MENSAL") <= 11))  # 12 meses ate M-1
        .withColumn("cod_idef_pess", F.col("cod_idef_pess"))
        .withColumn("DATREF_DEFAS", datref_defas)
        .withColumn("quantidade", F.lit(1))
        .withColumn("PK_DATREF_XX", pk_datref)
    )

    for c, v in flags.items():
        tmp = tmp.withColumn(c, F.when(cls == F.lit(v), F.lit(1)).otherwise(F.lit(0)))

    # BUG FIX: added missing commas after "cod_idef_pess" and "v1_transacao_mes"
    # (missing commas silently concatenate adjacent string literals in Python)
    return tmp.select(
        "cod_idef_pess",
        "DATREF_DEFAS",
        "ID_MENSAL",
        "dt_anomes",
        "v1_transacao_mes",
        "categoria_macro",
        "quantidade",
        *list(flags.keys()),
        "PK_DATREF_XX"
    )

In [ ]:
def table_exists(fullname: str) -> bool:
    try:
        return spark.catalog.tableExists(fullname)
    except Exception:
        # fallback (algumas versoes aceitam (db, table))
        db, tb = fullname.split(".", 1)
        return spark.catalog.tableExists(db, tb)

# BUG FIX: 'spark.sq1' (digit 1) -> 'spark.sql'
spark.sql(f"CREATE DATABASE IF NOT EXISTS {OUT_DB}")

if not table_exists(OUT_FULLNAME):
    # cria o schema com um DF vazio (limit(0)) e registra no Glue Catalog
    df_schema = build_month_df("202603").limit(0)
    (
        df_schema.write
            .mode("overwrite")
            .format("parquet")
            .option("path", OUT_S3_BASE)
            .partitionBy("PK_DATREF_XX")
            # BUG FIX: '.SaveAsTable' -> '.saveAsTable' (Python is case-sensitive)
            .saveAsTable(OUT_FULLNAME)
    )
    print("Tabela criada no Glue Catalog:", OUT_FULLNAME)

In [ ]:
for anomes_param in refs:
    df_out = build_month_df(anomes_param)
    (
        df_out
        # PERF FIX: removed .repartition(64) — it adds an unnecessary extra shuffle;
        # maxRecordsPerFile already controls output file size
        .write
        .mode("overwrite")
        .format("parquet")
        # PERF FIX: removed overwrite=True from insertInto — redundant with
        # partitionOverwriteMode=dynamic already set in spark.conf
        .insertInto(OUT_FULLNAME)
    )
    # PERF FIX: unpersist each month's plan after writing to free executor memory
    # between iterations and prevent plan accumulation over 24 sequential jobs
    df_out.unpersist()
    print("OK mes:", anomes_param)

src.unpersist()
print("FINALIZADO: particoes geradas de 202404 ate 202603 em", OUT_FULLNAME)